# Module 09: asyncio for Java Developers

## Interview Pro-Tip

**"How do you handle concurrent I/O in Python without threads?"**

`asyncio` is Python's answer to reactive/event-loop concurrency. If you've worked with
Node.js, Netty, or CompletableFuture chains, the mental model is similar — single-threaded,
cooperative multitasking with an event loop.

## Java Recall

```java
// Java: CompletableFuture for async composition
CompletableFuture<String> f1 = CompletableFuture.supplyAsync(() -> fetch(url1));
CompletableFuture<String> f2 = CompletableFuture.supplyAsync(() -> fetch(url2));

CompletableFuture.allOf(f1, f2).join();
List<String> results = Stream.of(f1, f2)
    .map(CompletableFuture::join)
    .collect(toList());
```

In Java, `CompletableFuture` runs work on the ForkJoinPool (threads). Virtual threads
(Java 21+) make this even more lightweight — but it's still threads underneath.

## Python Way

```python
import asyncio

# async/await — keywords that define cooperative coroutines
async def fetch(url: str) -> str:
    await asyncio.sleep(0.1)  # simulates I/O — yields control
    return f"data_{url}"

# Run multiple coroutines CONCURRENTLY (not in parallel)
async def main():
    results = await asyncio.gather(
        fetch("http://a.com"),
        fetch("http://b.com"),
    )
    return results  # ['data_http://a.com', 'data_http://b.com']

# Entry point — start the event loop
asyncio.run(main())
```

**Key insight:** `await` is an explicit yield point. The coroutine says "I'm waiting
for I/O, someone else can run." This is cooperative, not preemptive. If a coroutine
never awaits, it blocks the entire event loop — just like blocking the event loop
in Node.js.

## Key Differences

| Concept | Java | Python |
|---------|------|--------|
| Async primitive | `CompletableFuture<T>` | `Coroutine` (async def function) |
| Await equivalent | `.get()` / `.join()` | `await` keyword |
| Running in parallel | `allOf().join()` | `asyncio.gather(*tasks)` |
| Event loop | Hidden (ForkJoinPool) | Explicit — `asyncio.run()` |
| Threads involved? | Yes (ForkJoinPool threads) | No — single-threaded event loop |
| Cancellation | `future.cancel(true)` | `task.cancel()` — raises CancelledError |
| Async iteration | Reactive Streams / Flow API | `async for` + async generators |
| Task creation | `supplyAsync(() -> ...)` | `asyncio.create_task(coro)` |

**When to use asyncio vs threading vs multiprocessing:**

| Workload | Use |
|----------|-----|
| I/O-bound, high concurrency (1000s of connections) | `asyncio` |
| I/O-bound, moderate concurrency, existing sync code | `ThreadPoolExecutor` |
| CPU-bound | `ProcessPoolExecutor` |
| Mixed CPU + I/O | `asyncio` + `run_in_executor()` for CPU parts |

## Pitfalls and Interview Traps

### 1. Forgetting `await`
```python
async def bug():
    asyncio.sleep(1)  # BUG! Returns a coroutine, doesn't execute it
    # RuntimeWarning: coroutine 'sleep' was never awaited
```

### 2. Blocking the event loop
```python
async def bad():
    time.sleep(5)  # BLOCKS the event loop! Use await asyncio.sleep(5)
```

### 3. Mixing sync and async
You cannot call `await` inside a regular `def` function. Async is **viral** —
once you go async, your callers must be async too (or use `asyncio.run()`).

### 4. `asyncio.gather()` vs `asyncio.create_task()`
- `gather()` — run coroutines concurrently, wait for all, return results in order
- `create_task()` — schedule a coroutine to run, get a Task back, await it later

## Interview Answer Template

**Q: "How do you run concurrent I/O operations in Python?"**

A: "Python has three concurrency models: threading for simple I/O, asyncio for
high-concurrency I/O with cooperative multitasking, and multiprocessing for CPU-bound
work. For I/O-bound workloads with many concurrent operations, asyncio is preferred —
it's single-threaded, uses an event loop, and coroutines explicitly yield control with
`await`. The key primitive is `asyncio.gather()` to run tasks concurrently. Unlike
Java's CompletableFuture which uses threads, asyncio coroutines all run on one thread
— no context switching overhead, but you must never block the event loop."

## Your Practice

Open `initial/practice.py`. Implement:

1. **async_fetch_all** — Concurrent async fetches with `asyncio.gather()`
2. **async_countdown** — Async generator (combines `async def` + `yield`)
3. **run_concurrently** — Generic `asyncio.gather()` wrapper
4. **async_timer** — Measure coroutine wall-clock time

Run: `python run_test.py initial` / `python run_test.py complete`.

## References

- [asyncio — Asynchronous I/O](https://docs.python.org/3/library/asyncio.html)
- [Coroutines and Tasks](https://docs.python.org/3/library/asyncio-task.html)
- [PEP 492 — async/await syntax](https://peps.python.org/pep-0492/)
- [PEP 525 — Asynchronous Generators](https://peps.python.org/pep-0525/)